In [1]:
!pip install selectolax
!pip install playwright

In [2]:
!playwright install chromium

In [ ]:
import time
import random
import asyncio
from urllib.parse import urlparse
from playwright.async_api import async_playwright

class HeadlessBrowserManager:
    def __init__(self):
        self.playwright: Playwright | None = None
        self.browser: Browser | None = None

    async def initialize(self):
        """launches a localize headless chromium browser instance"""
        if not self.browser:
            self.playwright = await async_playwright().start()
            self.browser = await self.playwright.chromium.launch(
                headless=False,
                args=["--disable-gpu", "--no-sandbox", "--disable-dev-shm-usage"],
            )

    async def scrape_dynamic_page(self, url: str, wait_for_selector: str = "p", use_page = False):
        """Loads a page in a full browser environment, waits for elements to render, and pulls HTML."""

        await self.initialize()

        if self.browser:
            # Open an isolated browser tab context with desktop screen dimension mimicry
            context = await self.browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                viewport={"width": 1280, "height": 800},
            )

            page = await context.new_page()
            
            try:
                # Navigate to the target web novel page URL
                await page.goto(url, wait_until="domcontentloaded", timeout=30000)

                # Explicitly wait for javascript text frameworks to finish rendering elements into DOM
                await page.wait_for_selector(wait_for_selector, timeout=10000)

                # Extract the raw rendered HTML string code format
                html_content = await page.content()
                return (html_content, page if use_page else None)

            finally:
                if not use_page:
                    await page.close()
                    await context.close()

                
    async def fetch_with_browser_interaction(
        self, url: str, *, wait_for_selector="body", interaction_callback=None
    ) -> str | None:
        """
        Launches Playwright, navigates to the page, executes an optional
        custom callback (like clicking tabs), and harvests the final DOM.
        """
        await self.initialize()

        if self.browser:
            context = await self.browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                viewport={"width": 1280, "height": 800},
            )
            page = await context.new_page()

            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                # Explicitly wait for javascript text frameworks to finish rendering elements into DOM
                await page.wait_for_selector(wait_for_selector, timeout=10000)

                # If the specific scraper passed a tab-toggling instruction, execute it now
                if interaction_callback:
                    await interaction_callback(page)
                    # Give JavaScript a brief window to animate or hydrate the new layout
                    await page.wait_for_timeout(800)

                return await page.content()
            finally:
                await page.close()
                await context.close()

    async def shutdown(self):
        """Closes browser background processes cleany on app exit."""
        if self.browser:
            await self.browser.close()
        if self.playwright:
            await self.playwright.stop()


test_web_novel = "https://wtr-lab.com/en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin"

async def fetch_page_content():
    browser = HeadlessBrowserManager()
    content = await browser.scrape_dynamic_page(test_web_novel, wait_for_selector=".chapter-details")
    return content


async def fetch_main_page_html_contents(url: str):
    instance = HeadlessBrowserManager()

    await instance.initialize()
    browser = instance.browser

    initial_html_content: str = ''
    chapters: list[str] = []

    if browser:
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 800},
        )

        page = await context.new_page()

        try:
            # Navigate to the target web novel page URL
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)

            # Explicitly wait for javascript text frameworks to finish rendering elements into DOM
            await page.wait_for_selector(".chapter-details", timeout=60000)

            initial_html_content = await page.content()

            tab = page.get_by_role("tab", name="Table of Contents")
            print(f"ToC {tab}")
            await asyncio.sleep(1)
            await tab.click()
            await page.wait_for_selector(".chapter-list", timeout=10000)


            accordion_items = page.locator("div[data-slot='accordion-item']")
            count = await accordion_items.count()
            print(f"[+] Found {count} accordion panels to expand.")
            
            accordions = await accordion_items.all()
            
            if count > 0:
                for idx, item in enumerate(accordions):
                    # 1. Fixed the quote nesting issue
                    data_index = await item.get_attribute("data-index")
                    print(f"[+] Processing accordion index: {data_index} (Loop index: {idx})")
                    
                    # Click the trigger to open the accordion
                    trigger = item.locator("button[data-slot='accordion-trigger']")
                    await trigger.click()
            
                    # Target the specific anchor tags inside this item
                    anchor_elements = item.locator("div[data-slot='accordion-content'] a")
                    
                    # FIX: Explicitly wait for the first link in THIS accordion to become attached/visible.
                    # This pauses execution until the network/animation finishes rendering the links.
                    try:
                        await anchor_elements.first.wait_for(state="attached", timeout=5000)
                    except Exception:
                        print(f"[!] Warning: No links appeared in accordion {idx} after 5 seconds.")
                    
                    for a in await anchor_elements.all():
                        link = await a.get_attribute("href")
                        if link:
                            chapters.append(link)
                            
                    await asyncio.sleep(1)
            
            return initial_html_content, chapters

        finally:
            await page.close()
            await context.close()
            await instance.shutdown()

# result = await fetch_page_content()
# table_of_contents_result = await fetch_chapter_tab_content()

# table_of_contents_result

results = await fetch_main_page_html_contents(test_web_novel)

info_html = results[0]
raw_chapter_links = results[1]

print(f"\n\n[+] Found {len(raw_chapter_links)} Chapters")



In [ ]:
from urllib.parse import urlsplit
from selectolax.lexbor import LexborHTMLParser

tree = LexborHTMLParser(info_html)

parsed = urlparse(test_web_novel)
base_url = f"{parsed.scheme}://{parsed.netloc}"

base_url

In [34]:

titlebox = tree.css_first(".p-2[data-slot='card-content']").child
statusbox = tree.css_first(".p-2[data-slot='card-content']").child.next
tagbox = tree.css_first(".p-2[data-slot='card-content']").last_child
detailbox = tree.css_first(".p-2[data-slot='card-content'].chapter-details")

In [35]:
title = titlebox.css_first("h1").text()
other_title = titlebox.css_first("p").text()

(title, other_title)

('Lord: God-tier Attribute, Recruits Fallen Angels of Original Sin',
 '领主：神级词条，招募原罪堕天使')

In [36]:
cover_image_url = statusbox.css_first("img").attributes['src']
total_chapters = statusbox.css_first("span[translate='no']:lexbor-contains('chapters'i)").next.text()

(cover_image_url, total_chapters)

('/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FujGWOOGMtucuE5A8XnGYu1cBWtQClJi3k8BzzzVOh_4.jpeg&w=344',
 '664')

In [37]:
tags = [tag.text().lower() for tag in tagbox.css("span")]

tags

['male',
 'kingdom building',
 'reincarnation',
 'system',
 'game elements',
 'class awakening',
 'action',
 'fantasy',
 'game',
 'harem',
 'urban-life']

In [38]:
summary = detailbox.css_first(".desc-wrap > span.description").text()
summary

"(The rating just came out and will rise later. It's definitely a satisfying read, not toxic or frustrating, and it has a bit of logic!)[Lord] + [Troop Type] + [Conquest] + [Multiple Women] + [Construction] + [Hoarding Rat]Su Ye transmigrated to a world where everyone is a lord, and being a lord is everything, as well as the path to godhood.Starting Awakening Talent: Supreme Ruler, First Trait: Echo of Strong Luck.Echoes of Fortune: Upgrading Lord's Heart guarantees a gold or higher-level affix.Obtained terms: Super God Evolution (Red), Crystal Palace (Gold)...Super God Evolution: Can evolve any barracks into a god-level barracks, with unit loyalty remaining constant at 100 and never decreasing.Crystal Palace: This territory can only recruit female units. Female units have a 100% increase in combat power and a potential level of +1!...........Years later, fallen angels chanted the praises of the God-Emperor, illuminating all worlds. Primordial dragons roamed the world, amidst countless

In [39]:
detailgrid = detailbox.css_first("[data-slot='tabs-content']").css_first("span:lexbor-contains('details'i)").parent.next
status = detailgrid.css_first(":lexbor-contains('status'i)").next.text()
author = detailgrid.css_first(":lexbor-contains('author'i)").next.child.text()
other_author_name = detailgrid.css_first(":lexbor-contains('author'i)").next.last_child.text()

print(f"Status: {status.lower()}\n", f"Author: {author}, {other_author_name}", sep="")

Status: ongoing
Author: 浮生老五, Fu Sheng Lao Wu


In [ ]:
# Clean up the raw chapter links

links = [f"{base_url}/{x}" for x in raw_chapter_links]
links

In [71]:

def parse_chapter(url):
    instance = HeadlessBrowserManager()
    def toggle_ai(page):
        page.wait_for_selector(".sdsd", timeout=60000)

    html_content = instance.fetch_with_browser_interaction(url, wait_for_selector=".chapter-body")

    tree = LexborHTMLParser(html_content)
    
    
        